In [1]:
from summer3 import lazyprops as slp
from summer3.examples import get_tb_cmap
import polars as pl
import numpy as np

In [2]:
cmap = get_tb_cmap()

In [3]:
pt = slp.build_property_tables(cmap, cmap.compartments)

In [ ]:
class PropertyCMap:
    def __init__(self, cmap, prop_table: slp.PropertyTable):
        self.cmap = cmap
        self.prop_table = prop_table

    @classmethod
    def from_cmap(cls, cmap):
        return cls(cmap, slp.build_property_tables(cmap, cmap.compartments))
    
    def filter(self, expr):
        pt = self.prop_table
        query_expr = expr.actualize(pt)

        filtered_df = pt.table.filter(query_expr)

        #validity_s = filtered_df.select(~pl.all().eq(-1).all())

        #valid_columns = [c.name for c in validity_s if c[0]]
        #valid_columns = [k for k,v in validity.items() if v]

        #valid_df = filtered_df[valid_columns]

        # Pass valid_df to erase stratifications, or filtered_df to retain all
        new_pt = pt.filter(filtered_df)

        return PropertyCMap(cmap[filtered_df["index"].to_numpy()], new_pt)
    
    def get_accessors(self):
        return list(self.prop_table.properties.values())
    
    def __repr__(self):
        return f"[PCMap]{self.prop_table}"

In [12]:
pcmap = PropertyCMap.from_cmap(cmap)
history, age, early_tb, inf_status, clin_status = pcmap.get_accessors()

In [13]:
pt = pcmap.prop_table
pt

PropertyTable
shape: (27, 6)
┌───────────┬───────┬────────────┬──────────────┬───────────────┬───────┐
│ history_0 ┆ age_0 ┆ early_tb_0 ┆ inf_status_0 ┆ clin_status_0 ┆ index │
│ ---       ┆ ---   ┆ ---        ┆ ---          ┆ ---           ┆ ---   │
│ str       ┆ str   ┆ str        ┆ str          ┆ str           ┆ i64   │
╞═══════════╪═══════╪════════════╪══════════════╪═══════════════╪═══════╡
│ naive     ┆ 0     ┆ null       ┆ null         ┆ null          ┆ 0     │
│ naive     ┆ 15    ┆ null       ┆ null         ┆ null          ┆ 1     │
│ naive     ┆ 75    ┆ null       ┆ null         ┆ null          ┆ 2     │
│ early     ┆ 0     ┆ incipient  ┆ null         ┆ null          ┆ 3     │
│ early     ┆ 0     ┆ contained  ┆ null         ┆ null          ┆ 4     │
│ …         ┆ …     ┆ …          ┆ …            ┆ …             ┆ …     │
│ active    ┆ 75    ┆ null       ┆ inf          ┆ subclin       ┆ 22    │
│ active    ┆ 75    ┆ null       ┆ inf          ┆ clin          ┆ 23    │
│ recover

In [14]:
history.prop_key.strata

('naive', 'early', 'active', 'recovery')

In [17]:
pcmap.filter(history == "early")

[PCMap]PropertyTable
shape: (9, 6)
┌───────────┬───────┬────────────┬──────────────┬───────────────┬───────┐
│ history_0 ┆ age_0 ┆ early_tb_0 ┆ inf_status_0 ┆ clin_status_0 ┆ index │
│ ---       ┆ ---   ┆ ---        ┆ ---          ┆ ---           ┆ ---   │
│ str       ┆ str   ┆ str        ┆ str          ┆ str           ┆ i64   │
╞═══════════╪═══════╪════════════╪══════════════╪═══════════════╪═══════╡
│ early     ┆ 0     ┆ incipient  ┆ null         ┆ null          ┆ 0     │
│ early     ┆ 0     ┆ contained  ┆ null         ┆ null          ┆ 1     │
│ early     ┆ 0     ┆ cleared    ┆ null         ┆ null          ┆ 2     │
│ early     ┆ 15    ┆ incipient  ┆ null         ┆ null          ┆ 3     │
│ early     ┆ 15    ┆ contained  ┆ null         ┆ null          ┆ 4     │
│ early     ┆ 15    ┆ cleared    ┆ null         ┆ null          ┆ 5     │
│ early     ┆ 75    ┆ incipient  ┆ null         ┆ null          ┆ 6     │
│ early     ┆ 75    ┆ contained  ┆ null         ┆ null          ┆ 7     │
│ e

In [20]:
early_tb.prop_key.strata

('incipient', 'contained', 'cleared')

In [23]:
early_tb.prop_key.categories(), inf_status.prop_key.categories()

(CategoryGroup
 Category: [(Stratification: early_tb, ['incipient'])]
 Category: [(Stratification: early_tb, ['contained'])]
 Category: [(Stratification: early_tb, ['cleared'])],
 CategoryGroup
 Category: [(Stratification: inf_status, ['noninf'])]
 Category: [(Stratification: inf_status, ['inf'])])

In [24]:
pcmap.filter(history == "early")

[PCMap]PropertyTable
shape: (9, 6)
┌───────────┬───────┬────────────┬──────────────┬───────────────┬───────┐
│ history_0 ┆ age_0 ┆ early_tb_0 ┆ inf_status_0 ┆ clin_status_0 ┆ index │
│ ---       ┆ ---   ┆ ---        ┆ ---          ┆ ---           ┆ ---   │
│ str       ┆ str   ┆ str        ┆ str          ┆ str           ┆ i64   │
╞═══════════╪═══════╪════════════╪══════════════╪═══════════════╪═══════╡
│ early     ┆ 0     ┆ incipient  ┆ null         ┆ null          ┆ 0     │
│ early     ┆ 0     ┆ contained  ┆ null         ┆ null          ┆ 1     │
│ early     ┆ 0     ┆ cleared    ┆ null         ┆ null          ┆ 2     │
│ early     ┆ 15    ┆ incipient  ┆ null         ┆ null          ┆ 3     │
│ early     ┆ 15    ┆ contained  ┆ null         ┆ null          ┆ 4     │
│ early     ┆ 15    ┆ cleared    ┆ null         ┆ null          ┆ 5     │
│ early     ┆ 75    ┆ incipient  ┆ null         ┆ null          ┆ 6     │
│ early     ┆ 75    ┆ contained  ┆ null         ┆ null          ┆ 7     │
│ e

In [18]:
pcmap.filter(history == "active")

[PCMap]PropertyTable
shape: (12, 6)
┌───────────┬───────┬────────────┬──────────────┬───────────────┬───────┐
│ history_0 ┆ age_0 ┆ early_tb_0 ┆ inf_status_0 ┆ clin_status_0 ┆ index │
│ ---       ┆ ---   ┆ ---        ┆ ---          ┆ ---           ┆ ---   │
│ str       ┆ str   ┆ str        ┆ str          ┆ str           ┆ i64   │
╞═══════════╪═══════╪════════════╪══════════════╪═══════════════╪═══════╡
│ active    ┆ 0     ┆ null       ┆ noninf       ┆ subclin       ┆ 0     │
│ active    ┆ 0     ┆ null       ┆ noninf       ┆ clin          ┆ 1     │
│ active    ┆ 0     ┆ null       ┆ inf          ┆ subclin       ┆ 2     │
│ active    ┆ 0     ┆ null       ┆ inf          ┆ clin          ┆ 3     │
│ active    ┆ 15    ┆ null       ┆ noninf       ┆ subclin       ┆ 4     │
│ …         ┆ …     ┆ …          ┆ …            ┆ …             ┆ …     │
│ active    ┆ 15    ┆ null       ┆ inf          ┆ clin          ┆ 7     │
│ active    ┆ 75    ┆ null       ┆ noninf       ┆ subclin       ┆ 8     │
│ 

In [16]:
pcmap.filter(history.is_between("early","active") & (age=="15"))

[PCMap]PropertyTable
shape: (7, 6)
┌───────────┬───────┬────────────┬──────────────┬───────────────┬───────┐
│ history_0 ┆ age_0 ┆ early_tb_0 ┆ inf_status_0 ┆ clin_status_0 ┆ index │
│ ---       ┆ ---   ┆ ---        ┆ ---          ┆ ---           ┆ ---   │
│ str       ┆ str   ┆ str        ┆ str          ┆ str           ┆ i64   │
╞═══════════╪═══════╪════════════╪══════════════╪═══════════════╪═══════╡
│ early     ┆ 15    ┆ incipient  ┆ null         ┆ null          ┆ 0     │
│ early     ┆ 15    ┆ contained  ┆ null         ┆ null          ┆ 1     │
│ early     ┆ 15    ┆ cleared    ┆ null         ┆ null          ┆ 2     │
│ active    ┆ 15    ┆ null       ┆ noninf       ┆ subclin       ┆ 3     │
│ active    ┆ 15    ┆ null       ┆ noninf       ┆ clin          ┆ 4     │
│ active    ┆ 15    ┆ null       ┆ inf          ┆ subclin       ┆ 5     │
│ active    ┆ 15    ┆ null       ┆ inf          ┆ clin          ┆ 6     │
└───────────┴───────┴────────────┴──────────────┴───────────────┴───────┘

In [44]:
age.prop_key.strata

('0', '15', '75')

In [42]:
pcmap.filter(age < "15")

In [39]:
class DerivedAccessKey:
    def __init__(self, k, modifier):
        self.k = k
        self.modifier = modifier

    def __hash__(self):
        return hash((self.k,self.modifier))
    
    def __repr__(self):
        return f"{self.modifier}[{self.k}]"

In [41]:
DerivedAccessKey(history, "dest")

dest[PropertyAccessor[Stratification: history]]

In [9]:
pcmap.prop_table.properties

{Stratification: history: PropertyAccessor[Stratification: history],
 Stratification: age: PropertyAccessor[Stratification: age],
 Stratification: early_tb: PropertyAccessor[Stratification: early_tb],
 Stratification: inf_status: PropertyAccessor[Stratification: inf_status],
 Stratification: clin_status: PropertyAccessor[Stratification: clin_status]}

In [ ]:
pcmap.prop_table.table.with_columns(index=np.arange(5,len(pcmap.prop_table.table)+5))

history_0,age_0,early_tb_0,inf_status_0,clin_status_0,index
i64,i64,i64,i64,i64,i64
0,0,null,null,null,5
0,1,null,null,null,6
0,2,null,null,null,7
1,0,0,null,null,8
1,0,1,null,null,9
…,…,…,…,…,…
2,2,null,1,0,27
2,2,null,1,1,28
3,0,null,null,null,29


In [ ]:
((age == "15") or (history == "naive"))

In [ ]:
from bidict import bidict

In [ ]:
from typing import Hashable

UnameIndexKeyMap = dict[str, bidict[int, str]]
UnameObjectMap = dict[str, Hashable]


In [ ]:
pcmap.prop_table.uname_strat_map

bidict({'history_0': Stratification: history, 'age_0': Stratification: age, 'early_tb_0': Stratification: early_tb, 'inf_status_0': Stratification: inf_status, 'clin_status_0': Stratification: clin_status})

In [ ]:
pcmap.filter((age=='0') & (history == "naive")).prop_table

history_0,age_0,index
i64,i64,i64
0,0,0


In [ ]:
pcmap.filter(((age == "15") & ((inf_status == "noninf") | (inf_status.is_null())))).cmap

CompartmentContainer view of 0x2206253922832:
array([Compartment :[(Stratification: history, 'naive'), (Stratification: age, '15')],
       Compartment :[(Stratification: history, 'early'), (Stratification: age, '15'), (Stratification: early_tb, 'incipient')],
       Compartment :[(Stratification: history, 'early'), (Stratification: age, '15'), (Stratification: early_tb, 'contained')],
       Compartment :[(Stratification: history, 'early'), (Stratification: age, '15'), (Stratification: early_tb, 'cleared')],
       Compartment :[(Stratification: history, 'active'), (Stratification: age, '15'), (Stratification: inf_status, 'noninf'), (Stratification: clin_status, 'subclin')],
       Compartment :[(Stratification: history, 'active'), (Stratification: age, '15'), (Stratification: inf_status, 'noninf'), (Stratification: clin_status, 'clin')],
       Compartment :[(Stratification: history, 'recovery'), (Stratification: age, '15')]],
      dtype=object)array([ 1,  6,  7,  8, 16, 17, 25])

In [ ]:
age.prop_key.categories()

CategoryGroup
Category: [(Stratification: age, ['0'])]
Category: [(Stratification: age, ['15'])]
Category: [(Stratification: age, ['75'])]

In [ ]:
age.prop_key.strata[1:]

('15', '75')

In [ ]:
def filter_strat(astrat, expr):
    ftable = astrat.ptable.table.filter(expr.actualize(astrat.ptable))
    slp.AccessorStrat()
    strat = expr.pa.prop_key
    return strat

In [ ]:
astrat = slp.build_astrat("age", pl.Series([str(i) for i in range(32768)]))

In [ ]:
astrat.ptable.table.filter((astrat > "15").actualize(astrat.ptable))

age_0,index
i64,i64
16,16
17,17
18,18
19,19
20,20
…,…
32763,32763
32764,32764
32765,32765


In [ ]:
pl.Series(["0","15"])

""
str
"""0"""
"""15"""


In [ ]:
class NewStrat:
    def __init__(self, name, strata, table):
        self.name = name
        self.strata = strata
        self.table = table

    @classmethod
    def new(cls, name, strata):
        strat = cls(name, strata, None)
        table = slp.strat_to_prop_table(strat)
        strat.table = table


In [ ]:
agept = slp.strat_to_prop_table(age.prop_key)
accessor = age #list(agept.properties.values())[0]

In [ ]:
def filter_pt(pt, expr):
    query_expr = expr.actualize(pt)
    fpt = pt.table.filter(query_expr)

    validity_s = fpt.select(~pl.all().eq(-1).all())

    valid_columns = [c.name for c in validity_s if c[0]]
    #valid_columns = [k for k,v in validity.items() if v]

    valid_pt = fpt[valid_columns]

    return valid_pt

In [ ]:
age.prop_key.strata[(0,1)]

TypeError: tuple indices must be integers or slices, not tuple

In [ ]:
age.prop_key.strata[filter_pt(agept, accessor >= "15")["index"].to_list()]

TypeError: tuple indices must be integers or slices, not list

In [ ]:
(age >= "15").actualize(agept)["index"]

TypeError: 'Expr' object is not subscriptable

In [ ]:
pcmap.filter((age >= "15") & (history > "naive")).cmap

CompartmentContainer view of 0x1599914834448:
array([Compartment :[(Stratification: history, 'early'), (Stratification: age, '15'), (Stratification: early_tb, 'incipient')],
       Compartment :[(Stratification: history, 'early'), (Stratification: age, '15'), (Stratification: early_tb, 'contained')],
       Compartment :[(Stratification: history, 'early'), (Stratification: age, '15'), (Stratification: early_tb, 'cleared')],
       Compartment :[(Stratification: history, 'early'), (Stratification: age, '75'), (Stratification: early_tb, 'incipient')],
       Compartment :[(Stratification: history, 'early'), (Stratification: age, '75'), (Stratification: early_tb, 'contained')],
       Compartment :[(Stratification: history, 'early'), (Stratification: age, '75'), (Stratification: early_tb, 'cleared')],
       Compartment :[(Stratification: history, 'active'), (Stratification: age, '15'), (Stratification: inf_status, 'noninf'), (Stratification: clin_status, 'subclin')],
       Compartment :

In [ ]:
pt.properties

{Stratification: history: PropertyAccessor[Stratification: history],
 Stratification: age: PropertyAccessor[Stratification: age],
 Stratification: early_tb: PropertyAccessor[Stratification: early_tb],
 Stratification: inf_status: PropertyAccessor[Stratification: inf_status],
 Stratification: clin_status: PropertyAccessor[Stratification: clin_status]}

In [ ]:
type((age > "15").actualize(pt))

polars.expr.expr.Expr

In [ ]:
query = (age != "15") & (history == "naive")

query_expr = query.actualize(pt)
fpt = pt.table.filter(query_expr)

validity_s = fpt.select(~pl.all().eq(-1).all())
validity = {c.name:c[0] for c in validity_s}

valid_columns = [c.name for c in validity_s if c[0]]
#valid_columns = [k for k,v in validity.items() if v]

fpt[valid_columns]

history_0,age_0,index
i64,i64,i64
0,0,0
0,2,2


In [ ]:
history.prop_key.strata

('naive', 'early', 'active', 'recovery')

In [ ]:
query = ~(inf_status == "inf") & (history == "recovery")

query_expr = query.actualize(pt)
fpt = pt.table.filter(query_expr)

validity_s = fpt.select(~pl.all().eq(-1).all())
validity = {c.name:c[0] for c in validity_s}

valid_columns = [c.name for c in validity_s if c[0]]
#valid_columns = [k for k,v in validity.items() if v]

fpt[valid_columns]

history_0,age_0,index
i64,i64,i64
3,0,24
3,1,25
3,2,26
